# Measure one late-interaction model on ViDoRe DocVQA + InfoVQA

Encodes the two 500-page splits with a model of the OptiVision registry (or any checkpoint plus a
loader name), runs the fixed-pipeline study and the selection study on each split, and exports a
small results zip. This is how a registry entry moves from `untested` / `unverified` to `measured`.

**Before running**
1. *Settings -> Accelerator*: **GPU T4 x2**. *Internet*: **On**.
2. Set `MODEL` below (see `optivision models` for the registry). For a checkpoint that is not in
   the registry, also set `LOADER` (for example `colpali-engine:colqwen2.5`).
3. A `remote-code:*` model runs Python from the model repository inside this session. Read that
   code first, then set `TRUST_REMOTE_CODE = True`.
4. Click **Run All** and wait for `CHECK OK` under the check cell. Then stop the session and use
   *Save Version -> Save & Run All (Commit)*.
5. Download `results_<model>.zip` from the version's *Output* tab.

The check cell stops the run unless the model returns finite unit-norm vectors of one width and
the queries retrieve their own pages. A model that fails it is not loaded correctly by this
package: keep the log and do not use the numbers.

What the results are: in-sample development measurements on these two splits. They are not
held-out evidence and say nothing about another corpus.

In [ ]:
# parameters
MODEL = "vidore/colSmol-500M"   # a model id of the registry, or any checkpoint with LOADER set
LOADER = None                   # e.g. "colpali-engine:colqwen2.5"; None uses the registry's loader
TRUST_REMOTE_CODE = False       # True only after reading the model repository's code
REVISION = None                 # for remote-code models: None = the registry's pin, or a commit hash
DTYPE = "float32"
MULTI_GPU = True                # spread the model over both T4s (keeps float32 for 3B+ models)
SPLITS = ["docvqa", "infovqa"]
BRANCH = "main"                 # the branch or tag of optivision-rag to run (0.4.0 or later)
EXTRA_PIP = []                  # extra packages the model card asks for, e.g. ["einops"]
DEADLINE_HOURS = 11.25          # everything must be packaged by then (Kaggle stops at 12 h)

In [ ]:
import subprocess, sys, os, json, shutil, glob, time
T0 = time.time()
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
import torch
print("torch", torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
total, used, free = shutil.disk_usage("/tmp")
print(f"/tmp free: {free / 1e9:.0f} GB")
assert free > 30e9, "need > 30 GB free in /tmp for the weights and the vectors"

## 1. Install (the checkout must not be named `optivision`)

In [ ]:
WORK = "/kaggle/working"
REPO = f"{WORK}/optivision-rag"
DATA = "/tmp/optdata"
SHORT = MODEL.split("/")[-1]
RESULTS = f"{WORK}/results_{SHORT}"
os.environ["HF_HOME"] = "/tmp/hf"
!rm -rf {REPO} && git clone -q --depth 1 --branch {BRANCH} https://github.com/Daemon-VI/optivision-rag.git {REPO}
!cd {REPO} && git log --oneline -1
!pip install -q -e "{REPO}[bench,merge,vlm]" 2>&1 | tail -2
# Kaggle preinstalls torchao 0.10; peft refuses to work while an incompatible torchao is importable
!pip uninstall -y -q torchao
os.makedirs(f"{DATA}/vectors", exist_ok=True); os.makedirs(RESULTS, exist_ok=True)

sys.path.insert(0, f"{REPO}/src")
from optivision.adapters import MODELS
info = MODELS.get(MODEL)
loader = LOADER or (info.loader if info else None)
assert loader, "MODEL is not in the registry: set LOADER (see optivision.adapters.supported_loaders())"
print("registry entry:", info)
print("loader:", loader)
extra = list(EXTRA_PIP)
if loader == "sentence-transformers":
    extra.append("sentence-transformers[image]>=6.0.0")
if loader == "pylate":
    extra.append("pylate")
if loader.startswith("colpali-engine:"):
    extra += ["-U", "colpali-engine"]
if extra:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra], capture_output=True, text=True)
    print("pip install", " ".join(extra), "-> exit", r.returncode, r.stderr[-300:])
if loader.startswith("remote-code:"):
    assert TRUST_REMOTE_CODE, "this model runs its own code: read it, then set TRUST_REMOTE_CODE = True"

def remaining():
    return DEADLINE_HOURS * 3600 - (time.time() - T0)

def run(cmd, log, env=None, budget=False):
    """Run a step and stop the notebook if it fails. With budget=True the step is
    cut off at the deadline instead, and the notebook goes on to packaging."""
    t = time.time()
    timed_out, code = False, None
    with open(f"{RESULTS}/{log}", "w") as f:
        try:
            code = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, text=True,
                                  env={**os.environ, **(env or {})},
                                  timeout=max(60, remaining()) if budget else None).returncode
        except subprocess.TimeoutExpired:
            timed_out = True
    tail = open(f"{RESULTS}/{log}").read().splitlines()
    print("\n".join(tail[-25:]))
    status = "CUT OFF at the deadline" if timed_out else f"exit {code}"
    print(f"-- {log}: {status} after {(time.time() - t) / 60:.1f} min")
    if timed_out:
        with open(f"{RESULTS}/TIMED_OUT_{log}.txt", "w") as f:
            f.write(f"{log} was cut off after {(time.time() - t) / 3600:.2f} h\n")
        return False
    assert code == 0, f"{log} failed: see the lines above"
    return True

## 2. Check on a few pages, then encode both splits

`scripts/encode_vectors.py` runs `check_retriever` on the first pages of a split before the full
encode and writes nothing if it fails.

In [ ]:
DATASETS = {"docvqa": "vidore/docvqa_test_subsampled", "infovqa": "vidore/infovqa_test_subsampled"}
OPTIONS = ["--dtype", DTYPE, "--check-pairs", "12"]
if LOADER:
    OPTIONS += ["--loader", LOADER]
if MULTI_GPU:
    OPTIONS += ["--multi-gpu"]
if TRUST_REMOTE_CODE:
    OPTIONS += ["--trust-remote-code"]
if REVISION:
    OPTIONS += ["--revision", REVISION]
ENCODE = [sys.executable, f"{REPO}/scripts/encode_vectors.py", MODEL]
first = DATASETS[SPLITS[0]]
run([*ENCODE, first, "--out", "/tmp/optdata/check", "--limit", "36", *OPTIONS], "check.log")
text = open(f"{RESULTS}/check.log").read()
assert '"ok": true' in text, "check_retriever failed: the model is not loaded correctly by this package"
print("CHECK OK")

In [ ]:
for s in SPLITS:
    run([*ENCODE, DATASETS[s], "--out", f"{DATA}/vectors", *OPTIONS], f"encode_{s}.log")
TAGS = [f"{SHORT}_{DATASETS[s].split('/')[-1]}" for s in SPLITS]
!ls -la {DATA}/vectors
!rm -rf /tmp/hf    # the weights are no longer needed
print(f"elapsed {(time.time() - T0) / 3600:.2f} h, {remaining() / 3600:.2f} h left")

## 3. Analysis, on the GPU for scoring

For each split, `s13_wide_study.py` (baseline, fixed codec / token / dimension pipelines, two tiers,
Pareto frontier) and then `s7b_selection_rules.py` (the held-out calibration study on the default
search space). The fixed-pipeline study runs for both splits first, so a cut-off costs the
selection study, not both.

In [ ]:
ENV = {"OPTIVISION_DATA": DATA, "OPTIVISION_OUT": RESULTS, "OPTIVISION_SCORE_DEVICE": "cuda",
       "OPTIVISION_CACHE_BYTES": str(4 * 1024**3), "PYTHONUNBUFFERED": "1"}
DONE = {}
for tag in TAGS:
    DONE[f"s13:{tag}"] = remaining() > 600 and run(
        [sys.executable, f"{REPO}/scripts/universal_study/s13_wide_study.py", f"vec:{tag}"], f"s13_{tag}.log", ENV, budget=True)
    print(f"elapsed {(time.time() - T0) / 3600:.2f} h, {remaining() / 3600:.2f} h left")
for tag in TAGS:
    DONE[f"s7b:{tag}"] = remaining() > 600 and run(
        [sys.executable, f"{REPO}/scripts/universal_study/s7b_selection_rules.py", f"vec:{tag}"], f"s7b_{tag}.log", ENV, budget=True)
    print(f"elapsed {(time.time() - T0) / 3600:.2f} h, {remaining() / 3600:.2f} h left")
print(DONE)

## 4. Package the results

In [ ]:
for tag in TAGS:
    shutil.copy(f"{DATA}/vectors/{tag}_qrels.json", RESULTS)
with open(f"{RESULTS}/run_summary.json", "w") as f:
    json.dump({"model": MODEL, "loader": loader, "dtype": DTYPE, "splits": SPLITS, "done": DONE,
               "branch": BRANCH, "total_hours": round((time.time() - T0) / 3600, 2)}, f, indent=1)
!pip freeze > {RESULTS}/pip_freeze.txt
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv > {RESULTS}/gpu.txt
!cd {REPO} && git rev-parse HEAD > {RESULTS}/commit.txt
!rm -rf {REPO}
!find {RESULTS} -type f | sort
!cd {WORK} && rm -f results_{SHORT}.zip && zip -q -r results_{SHORT}.zip results_{SHORT} && ls -la results_{SHORT}.zip